# CNN-Based Skin Lesion Classification Demo

This notebook reconstructs the clean delivery version of the project. It is designed for local Jupyter or Google Colab and uses relative project paths instead of hard-coded Windows paths.

The goal is to demonstrate the completed Stage 1 workflow: dataset preparation, preprocessing, CNN model comparison, evaluation figures, confusion matrices, and a top-3 prediction demo. The current default deployment model is **ResNet50**, because it achieved the best test accuracy and macro F1-score among the evaluated models.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'ham10000.yaml'
SPLITS_CSV = PROJECT_ROOT / 'data' / 'processed' / 'splits.csv'
RUNS_DIR = PROJECT_ROOT / 'runs'
FIGURES_DIR = PROJECT_ROOT / 'docs' / 'figures'
DEMO_DIR = PROJECT_ROOT / 'docs' / 'demo'

MODEL_ORDER = ['mobilenetv3_small_100', 'efficientnet_b0', 'densenet121', 'resnet50']
MODEL_LABELS = {
    'mobilenetv3_small_100': 'MobileNetV3 Small',
    'efficientnet_b0': 'EfficientNet-B0',
    'densenet121': 'DenseNet121',
    'resnet50': 'ResNet50',
}

def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


## Dataset Loading

The project uses the HAM10000 / ISIC skin lesion image dataset. The prepared split file stores image paths, labels, lesion IDs, and the assigned train/validation/test split. The split is grouped by `lesion_id` where available to reduce duplicate-lesion leakage.

In [ ]:
if SPLITS_CSV.exists():
    splits = pd.read_csv(SPLITS_CSV)
    display(splits.head())
    display(pd.crosstab(splits['split'], splits['label']))
else:
    print('Split CSV not found. Run src.skinlesion.prepare_ham10000 first.')


## Preprocessing and Transforms

Images are resized to 224 x 224 pixels and normalized with ImageNet statistics. Training uses augmentation such as random horizontal/vertical flips, rotation, and brightness/contrast changes. Validation, testing, and demo prediction use deterministic resizing and normalization.

In [ ]:
try:
    from src.skinlesion.data import build_transform
    train_transform = build_transform(split='train', image_size=224)
    test_transform = build_transform(split='test', image_size=224)
    print('Training transform:', train_transform)
    print('Test transform:', test_transform)
except Exception as exc:
    print('Transform preview skipped:', exc)


## Model Architecture Setup

Four transfer-learning CNN architectures were compared: MobileNetV3 Small, EfficientNet-B0, DenseNet121, and ResNet50. All models use ImageNet pretrained weights during training and replace the final classifier with a seven-class output layer.

In [ ]:
model_setup = pd.DataFrame([
    {'Model': 'MobileNetV3 Small', 'timm_name': 'mobilenetv3_small_100', 'Role': 'lightweight baseline'},
    {'Model': 'EfficientNet-B0', 'timm_name': 'efficientnet_b0', 'Role': 'efficient baseline'},
    {'Model': 'DenseNet121', 'timm_name': 'densenet121', 'Role': 'strong comparison baseline'},
    {'Model': 'ResNet50', 'timm_name': 'resnet50', 'Role': 'default deployment model'},
])
display(model_setup)


## Training Workflow / Saved Histories

The notebook uses saved `history.json` and `test_metrics.json` files when available. Full training can be reproduced from the repository with:

```bash
python -m src.skinlesion.train --config configs/ham10000.yaml --all-models
python -m src.skinlesion.evaluate --config configs/ham10000.yaml --model resnet50 --checkpoint runs/resnet50/best.pt --split test
```

In [ ]:
rows = []
for model_name in MODEL_ORDER:
    metrics_path = RUNS_DIR / model_name / 'test_metrics.json'
    history_path = RUNS_DIR / model_name / 'history.json'
    if metrics_path.exists() and history_path.exists():
        metrics = read_json(metrics_path)
        history = read_json(history_path)
        best_epoch = max(history, key=lambda row: row['validation_macro_f1'])
        rows.append({
            'Model': MODEL_LABELS[model_name],
            'Best Epoch': best_epoch['epoch'],
            'Train Accuracy': best_epoch['train_accuracy'],
            'Validation Accuracy': best_epoch['validation_accuracy'],
            'Test Accuracy': metrics['accuracy'],
            'Macro Precision': metrics['macro_precision'],
            'Macro Recall': metrics['macro_recall'],
            'Macro F1': metrics['macro_f1'],
            'Weighted F1': metrics['weighted_f1'],
        })

results = pd.DataFrame(rows)
percent_cols = ['Train Accuracy', 'Validation Accuracy', 'Test Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Weighted F1']
display(results.style.format({col: '{:.2%}' for col in percent_cols}))


## Evaluation Figures

The generated report figures summarize the training process and model comparison. Run `python -m src.skinlesion.report_assets --runs-dir runs --output-dir docs/figures` to regenerate them.

In [ ]:
for figure_name in ['model_comparison.png', 'training_curves.png']:
    figure_path = FIGURES_DIR / figure_name
    if figure_path.exists():
        display(Image(filename=str(figure_path)))
    else:
        print(f'Missing figure: {figure_path}')


## Confusion Matrix Summaries

The confusion matrix summaries include both raw counts and row-normalized values. The row-normalized view can be interpreted as per-class recall.

In [ ]:
for figure_name in ['resnet50_confusion_matrix_summary.png', 'densenet121_confusion_matrix_summary.png']:
    figure_path = FIGURES_DIR / figure_name
    if figure_path.exists():
        display(Image(filename=str(figure_path)))
    else:
        print(f'Missing figure: {figure_path}')


## Top-3 Prediction Demo

The demo folder contains selected test images and saved top-3 outputs. These outputs are intended to make the presentation stable even if a live backend demo is unavailable.

In [ ]:
demo_json_files = sorted(DEMO_DIR.glob('prediction_*.json'))
for json_path in demo_json_files:
    print('\n', json_path.name)
    demo = read_json(json_path)
    print(json.dumps(demo, indent=2))


## Conclusion

ResNet50 is selected as the default deployment model because it achieved the strongest overall test accuracy (80.22%) and macro F1-score (69.03%) among the evaluated architectures. DenseNet121 remains a strong comparison baseline with very similar macro F1, while EfficientNet-B0 provides a useful middle point between compactness and performance. MobileNetV3 Small is the most lightweight baseline but has lower classification performance on this dataset.

This prototype is for educational demonstration and research support only. It is not a medical diagnosis system.